In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "out_data"
OUT_DIR.mkdir(exist_ok=True)

print("ROOT   :", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)


In [35]:
# --- load feature table produced earlier (110D + label y) ---
X_FILE = OUT_DIR / "binary_vosa_XPcoeff_110d_L2.csv"
if not X_FILE.exists():
    raise FileNotFoundError(
        f"Missing {X_FILE}\n"
        "Expected the same artifact created in the data-construction notebook.\n"
        "Make sure out_data/binary_vosa_XPcoeff_110d_L2.csv exists."
    )

df_all = pd.read_csv(X_FILE)

feat_cols = [c for c in df_all.columns if c.startswith("c")]

df_all["source_id"] = df_all["source_id"].astype("int64")
df_all["y"] = df_all["y"].astype(int)

print("Loaded:", X_FILE.name)
print("Shape :", df_all.shape)
print("Positives:", int(df_all["y"].sum()), "| Negatives:", int((1 - df_all["y"]).sum()))
df_all.head()


Loaded: binary_vosa_XPcoeff_110d_L2.csv
Shape : (2815, 112)
Positives: 558 | Negatives: 2257


,source_id,y,c000,c001,c002,c003,c004,c005,c006,c007,...,c100,c101,c102,c103,c104,c105,c106,c107,c108,c109
0,2680574649477853952,0,0.804624,-0.435171,0.134028,-0.066597,0.028423,0.020254,0.003899,0.007772,...,0.000316,0.000140,-0.000387,0.000188,-0.000097,-0.000175,0.000278,0.000077,0.000106,-1.106871e-05
1,2682107811068484224,0,0.797602,-0.456332,0.146802,-0.076859,0.034934,0.030742,0.000744,0.008423,...,-0.001036,0.000899,-0.000558,0.001016,0.000010,0.000139,0.000197,-0.000139,-0.000032,1.730024e-04
2,2682170350087374464,0,0.804692,-0.442007,0.133211,-0.070096,0.028057,0.019926,0.003477,0.003134,...,-0.000436,-0.000344,-0.000291,0.000252,-0.000179,0.000098,0.000048,-0.000037,-0.000142,3.476371e-08
3,2683585696429935488,0,0.802656,-0.442244,0.137975,-0.070075,0.029988,0.021373,0.001878,0.006463,...,0.000140,-0.000001,-0.000058,-0.000139,-0.000073,0.000021,0.000116,0.000010,0.000008,-2.816741e-05
4,2684009283284602880,0,0.800380,-0.450982,0.143769,-0.072916,0.035159,0.025500,0.001328,0.009039,...,0.000016,-0.000036,0.000092,0.000031,-0.000037,0.000048,0.000033,0.000069,-0.000002,-4.591849e-06


In [36]:
X = df_all[feat_cols].to_numpy(dtype=float)
y = df_all["y"].to_numpy(dtype=int)

# 60/20/20 split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE
)

print("Train:", X_tr.shape, "pos:", y_tr.sum(), "neg:", (1-y_tr).sum())
print("Val  :", X_va.shape, "pos:", y_va.sum(), "neg:", (1-y_va).sum())
print("Test :", X_te.shape, "pos:", y_te.sum(), "neg:", (1-y_te).sum())


Train: (1689, 110) pos: 335 neg: 1354
Val  : (563, 110) pos: 112 neg: 451
Test : (563, 110) pos: 111 neg: 452


In [37]:
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

def _metrics_from_probs(y_true, p_pos, thr):
    y_pred = (p_pos >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1   = (2*prec*sens)/(prec+sens) if (prec+sens) else 0.0
    youden = sens + spec - 1.0
    return sens, spec, prec, f1, youden

def pick_threshold(y_true, p_pos, criterion="youden", grid_size=200):
    # robust threshold grid: percentiles of predicted probs + endpoints
    qs = np.linspace(0.0, 1.0, grid_size)
    thr_grid = np.unique(np.quantile(p_pos, qs))
    thr_grid = np.clip(thr_grid, 0.0, 1.0)

    best = None
    for thr in thr_grid:
        sens, spec, prec, f1, youden = _metrics_from_probs(y_true, p_pos, thr)
        score = youden if criterion == "youden" else f1
        row = (score, thr, sens, spec, prec)
        if best is None or row[0] > best[0]:
            best = row

    score, thr, sens, spec, prec = best
    return {"thr": float(thr), "sens": sens, "spec": spec, "prec": prec, "score": score}

def prob_metrics(y_true, p_pos):
    # Probability-level metrics (threshold-free)
    eps = 1e-15
    p = np.clip(p_pos, eps, 1 - eps)
    return {
        "ROC AUC": float(roc_auc_score(y_true, p)),
        "PR AUC": float(average_precision_score(y_true, p)),
        "Brier": float(brier_score_loss(y_true, p)),
        "Log loss": float(log_loss(y_true, p)),
    }

def style_results(df):
    metric_cols = ["Sensitivity", "Specificity", "Precision"]
    fmt = {
        "Best C": "{:.4g}",
        "Best gamma": "{:.4g}",
        "Threshold": "{:.3f}",
        "Sensitivity": "{:.3f}",
        "Specificity": "{:.3f}",
        "Precision": "{:.3f}",
    }
    return (df.style
              .format(fmt, na_rep="")
              .background_gradient(cmap="viridis", subset=metric_cols)
           )

def style_prob_results(df):
    # For AUC columns: higher is better.
    # For Brier/Log loss: lower is better -> use reversed colormap.
    auc_cols  = ["ROC AUC", "PR AUC"]
    loss_cols = ["Brier", "Log loss"]

    fmt = {
        "Best C": "{:.4g}",
        "Best gamma": "{:.4g}",
        "ROC AUC": "{:.4f}",
        "PR AUC": "{:.4f}",
        "Brier": "{:.5f}",
        "Log loss": "{:.5f}",
    }

    return (df.style
              .format(fmt, na_rep="")
              .background_gradient(cmap="viridis", subset=auc_cols)
              .background_gradient(cmap="viridis_r", subset=loss_cols)
           )

print("Helper funcs ready (incl. probability metrics).")


Helper funcs ready (incl. probability metrics).


In [38]:
# log-spaced grids
C_GRID = np.logspace(-4, 5, 19) # 1e-4 ... 1e5

GAMMA_GRID = np.logspace(-7, 1, 17) # 1e-7 ... 1e1 (RBF only)

VARIANTS = [
    # Linear SVM (LinearSVC + calibration -> predict_proba)
    {"name": "Linear",                   "kind": "linear", "zscore": False, "balanced": False},
    {"name": "Linear, balanced",         "kind": "linear", "zscore": False, "balanced": True},
    {"name": "Linear + zscore",          "kind": "linear", "zscore": True,  "balanced": False},
    {"name": "Linear + zscore, balanced","kind": "linear", "zscore": True,  "balanced": True},

    # RBF SVM (SVC(probability=True) + scaling enforced)
    {"name": "RBF + zscore",             "kind": "rbf",    "zscore": True,  "balanced": False},
    {"name": "RBF + zscore, balanced",   "kind": "rbf",    "zscore": True,  "balanced": True},
]

print("Variants:", [v["name"] for v in VARIANTS])
print("C grid:", C_GRID[:3], "...", C_GRID[-3:])
print("gamma grid:", GAMMA_GRID[:3], "...", GAMMA_GRID[-3:])


Variants: ['Linear', 'Linear, balanced', 'Linear + zscore', 'Linear + zscore, balanced', 'RBF + zscore', 'RBF + zscore, balanced']
C grid: [0.0001     0.00031623 0.001     ] ... [ 10000.          31622.77660168 100000.        ]
gamma grid: [1.00000000e-07 3.16227766e-07 1.00000000e-06] ... [ 1.          3.16227766 10.        ]


In [39]:
def fit_predict_proba_linear(Xtr, ytr, Xva, C, balanced, zscore):
    base = LinearSVC(C=C, class_weight=("balanced" if balanced else None), max_iter=20000)
    clf = CalibratedClassifierCV(base, method="sigmoid", cv=5)  # produces predict_proba

    if zscore:
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    else:
        pipe = Pipeline([("clf", clf)])

    pipe.fit(Xtr, ytr)
    p_va = pipe.predict_proba(Xva)[:, 1]
    return pipe, p_va

def fit_predict_proba_rbf(Xtr, ytr, Xva, C, gamma, balanced):
    # RBF strongly benefits from scaling; we enforce zscore by design in variants
    svc = SVC(
        C=C, gamma=gamma, kernel="rbf",
        class_weight=("balanced" if balanced else None),
        probability=True
    )
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", svc)])
    pipe.fit(Xtr, ytr)
    p_va = pipe.predict_proba(Xva)[:, 1]
    return pipe, p_va

def search_variant_on_val(variant, criterion):
    best = None
    best_model = None

    if variant["kind"] == "linear":
        for C in C_GRID:
            model, p_va = fit_predict_proba_linear(
                X_tr, y_tr, X_va, C=C,
                balanced=variant["balanced"],
                zscore=variant["zscore"]
            )
            pick = pick_threshold(y_va, p_va, criterion=criterion)
            row = (pick["score"], C, None, pick, model)
            if best is None or row[0] > best[0]:
                best = row
                best_model = model

        score, bestC, bestG, pick, best_model = best
        return {"bestC": float(bestC), "bestG": None, "thr": pick["thr"], "model": best_model}

    elif variant["kind"] == "rbf":
        for C in C_GRID:
            for gamma in GAMMA_GRID:
                model, p_va = fit_predict_proba_rbf(
                    X_tr, y_tr, X_va, C=C, gamma=gamma,
                    balanced=variant["balanced"]
                )
                pick = pick_threshold(y_va, p_va, criterion=criterion)
                row = (pick["score"], C, gamma, pick, model)
                if best is None or row[0] > best[0]:
                    best = row
                    best_model = model

        score, bestC, bestG, pick, best_model = best
        return {"bestC": float(bestC), "bestG": float(bestG), "thr": pick["thr"], "model": best_model}

    else:
        raise ValueError("Unknown variant kind")

def evaluate_on_test_threshold(model, thr):
    p_te = model.predict_proba(X_te)[:, 1]
    sens, spec, prec, f1, youden = _metrics_from_probs(y_te, p_te, thr)
    return {"Sensitivity": sens, "Specificity": spec, "Precision": prec}

def evaluate_on_test_probs(model):
    p_te = model.predict_proba(X_te)[:, 1]
    return prob_metrics(y_te, p_te)

def _postprocess_gamma(df):
    # show gamma only when relevant
    df = df.copy()
    df.loc[~df["Variant"].str.contains("RBF"), "Best gamma"] = np.nan
    return df

# -------------------------
# 1) Youden selection -> TEST threshold table
# -------------------------
rows = []
sels_youden = {}  # keep fitted models/params for probability table
for v in VARIANTS:
    sel = search_variant_on_val(v, criterion="youden")
    met = evaluate_on_test_threshold(sel["model"], sel["thr"])
    rows.append({
        "Variant": v["name"],
        "Best C": sel["bestC"],
        "Best gamma": sel["bestG"],
        "Threshold": sel["thr"],
        **met
    })
    sels_youden[v["name"]] = sel

df_test_youden = _postprocess_gamma(pd.DataFrame(rows))

# -------------------------
# 2) F1 selection -> TEST threshold table
# -------------------------
rows = []
sels_f1 = {}
for v in VARIANTS:
    sel = search_variant_on_val(v, criterion="f1")
    met = evaluate_on_test_threshold(sel["model"], sel["thr"])
    rows.append({
        "Variant": v["name"],
        "Best C": sel["bestC"],
        "Best gamma": sel["bestG"],
        "Threshold": sel["thr"],
        **met
    })
    sels_f1[v["name"]] = sel

df_test_f1 = _postprocess_gamma(pd.DataFrame(rows))

# --- Benchmark comparison ---
BENCHMARK = {
    'Youden': {
        'Sensitivity': 0.878,
        'Specificity': 0.965,
        'Precision': 0.838,
    },
    'F1': {
        'Sensitivity': 0.818,
        'Specificity': 0.985,
        'Precision': 0.920,
    },
}

def delta_vs_benchmark(df, selection_label):
    metrics = ['Sensitivity', 'Specificity', 'Precision']
    base = BENCHMARK.get(selection_label, BENCHMARK.get('Youden', {m:0.0 for m in metrics}))
    deltas = df[['Variant'] + metrics].copy()
    for m in metrics:
        deltas[m] = deltas[m] - base[m]
    deltas.insert(0, 'Selection', selection_label)
    return deltas

# Show benchmark deltas FIRST
_delta_y = delta_vs_benchmark(df_test_youden, 'Youden')
_delta_f = delta_vs_benchmark(df_test_f1, 'F1')

max_abs_y = float(np.nanmax(np.abs(_delta_y[['Sensitivity','Specificity','Precision']].to_numpy())))
max_abs_f = float(np.nanmax(np.abs(_delta_f[['Sensitivity','Specificity','Precision']].to_numpy())))

print('\nΔ vs benchmark — Youden:')
display(
    _delta_y.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_y, vmax=max_abs_y)
)

print('\nΔ vs benchmark — F1:')
display(
    _delta_f.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_f, vmax=max_abs_f)
)

print('\nTEST — evaluated using VAL-picked (hyperparams, threshold) from Youden selection:')
display(style_results(df_test_youden.sort_values(['Specificity','Sensitivity','Precision'], ascending=False)))

print('\nTEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:')
display(style_results(df_test_f1.sort_values(['Precision','Sensitivity','Specificity'], ascending=False)))

# Probability metrics table (Youden-picked models)
rows = []
for v in VARIANTS:
    sel = sels_youden[v['name']]
    pm = evaluate_on_test_probs(sel['model'])
    rows.append({
        'Variant': v['name'],
        'Best C': sel['bestC'],
        'Best gamma': sel['bestG'],
        **pm
    })
df_prob_youden = _postprocess_gamma(pd.DataFrame(rows))

print('\nTEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked models:')
display(style_prob_results(df_prob_youden.sort_values(['PR AUC','ROC AUC'], ascending=False)))



Δ vs benchmark — Youden:


,Selection,Variant,Sensitivity,Specificity,Precision
0,Youden,Linear,-0.175,+0.013,+0.048
1,Youden,"Linear, balanced",-0.148,-0.003,-0.011
2,Youden,Linear + zscore,-0.094,-0.082,-0.217
3,Youden,"Linear + zscore, balanced",-0.067,-0.111,-0.261
4,Youden,RBF + zscore,-0.121,-0.071,-0.202
5,Youden,"RBF + zscore, balanced",-0.094,-0.020,-0.061



Δ vs benchmark — F1:


,Selection,Variant,Sensitivity,Specificity,Precision
0,F1,Linear,-0.115,-0.007,-0.034
1,F1,"Linear, balanced",-0.151,-0.003,-0.018
2,F1,Linear + zscore,-0.133,-0.000,-0.004
3,F1,"Linear + zscore, balanced",-0.160,-0.003,-0.019
4,F1,RBF + zscore,-0.151,-0.000,-0.006
5,F1,"RBF + zscore, balanced",-0.097,-0.020,-0.087



TEST — evaluated using VAL-picked (hyperparams, threshold) from Youden selection:


,Variant,Best C,Best gamma,Threshold,Sensitivity,Specificity,Precision
0,Linear,316.2,,0.341,0.703,0.978,0.886
1,"Linear, balanced",31.62,,0.245,0.730,0.962,0.827
5,"RBF + zscore, balanced",1,0.003162,0.225,0.784,0.945,0.777
4,RBF + zscore,3.162,0.0001,0.130,0.757,0.894,0.636
2,Linear + zscore,0.0003162,,0.122,0.784,0.883,0.621
3,"Linear + zscore, balanced",0.0003162,,0.106,0.811,0.854,0.577



TEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:


,Variant,Best C,Best gamma,Threshold,Sensitivity,Specificity,Precision
2,Linear + zscore,0.3162,,0.430,0.685,0.985,0.916
4,RBF + zscore,1e+05,1e-05,0.428,0.667,0.985,0.914
1,"Linear, balanced",316.2,,0.440,0.667,0.982,0.902
3,"Linear + zscore, balanced",3.162,,0.449,0.658,0.982,0.901
0,Linear,316.2,,0.341,0.703,0.978,0.886
5,"RBF + zscore, balanced",10,0.003162,0.358,0.721,0.965,0.833



TEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked models:


,Variant,Best C,Best gamma,ROC AUC,PR AUC,Brier,Log loss
0,Linear,316.2,,0.8780,0.8127,0.06888,0.26316
5,"RBF + zscore, balanced",1,0.003162,0.8905,0.8120,0.06965,0.27573
1,"Linear, balanced",31.62,,0.8717,0.8026,0.07363,0.27614
3,"Linear + zscore, balanced",0.0003162,,0.8610,0.7569,0.08542,0.30319
2,Linear + zscore,0.0003162,,0.8559,0.7496,0.08690,0.30781
4,RBF + zscore,3.162,0.0001,0.8457,0.7429,0.08836,0.31982
